# Coordinate Utilities
GPS coordinate tools: offset in opposite direction, distance between points, and GPS timestamp lookup.

In [ ]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import numpy as np
from pyproj import Geod

geod = Geod(ellps="WGS84")

## Helper functions

In [ ]:
def azimuth_from_two_points(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Forward azimuth (degrees) from point 1 → point 2."""
    fwd_az, _, _ = geod.inv(lon1, lat1, lon2, lat2)
    return fwd_az % 360.0


def point_opposite_direction(lat: float, lon: float, azimuth_deg: float, distance_m: float = 4.0):
    """Return the point `distance_m` metres away in the opposite direction of `azimuth_deg`."""
    opposite_azimuth = (azimuth_deg + 180.0) % 360.0
    lon_out, lat_out, _ = geod.fwd(lon, lat, opposite_azimuth, distance_m)
    return lat_out, lon_out


def offset_from_input(
    lat: float,
    lon: float,
    distance_m: float = 4.0,
    azimuth_deg: float = None,
    lat2: float = None,
    lon2: float = None,
):
    """
    Compute a point `distance_m` metres opposite to the given direction.

    Supply either:
      - azimuth_deg  : explicit bearing in degrees (0=north, 90=east)
      - lat2, lon2   : second coordinate; azimuth derived as point1 → point2

    Returns (lat_out, lon_out, azimuth_used)
    """
    if azimuth_deg is not None:
        az = azimuth_deg
    elif lat2 is not None and lon2 is not None:
        az = azimuth_from_two_points(lat, lon, lat2, lon2)
    else:
        raise ValueError("Provide either azimuth_deg OR (lat2, lon2).")

    lat_out, lon_out = point_opposite_direction(lat, lon, az, distance_m)
    return lat_out, lon_out, az


def distance_between_points(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Geodetic distance in metres between two WGS84 coordinates."""
    _, _, dist = geod.inv(lon1, lat1, lon2, lat2)
    return abs(dist)


def delta_xy(lat1: float, lon1: float, lat2: float, lon2: float) -> dict:
    """
    Decompose the vector from point 1 → point 2 into East/North components (metres).

    Uses the geodetic azimuth and distance on the WGS84 ellipsoid, then projects:
      dx (east)  = distance * sin(azimuth)
      dy (north) = distance * cos(azimuth)

    Returns a dict with keys: distance, azimuth, dx, dy
    """
    fwd_az, _, dist = geod.inv(lon1, lat1, lon2, lat2)
    dist = abs(dist)
    az_rad = np.radians(fwd_az)
    dx = dist * np.sin(az_rad)   # east  (negative = west)
    dy = dist * np.cos(az_rad)   # north (negative = south)
    return {"distance": dist, "azimuth": fwd_az % 360.0, "dx": dx, "dy": dy}

## Distance between two points

In [ ]:
lat_a, lon_a = 52.008995, 4.355610
lat_b, lon_b = 52.009059, 4.355549

t = delta_xy(lat_a, lon_a, lat_b, lon_b)

print(f"Point A  : lat={lat_a:.10f}  lon={lon_a:.10f}")
print(f"Point B  : lat={lat_b:.10f}  lon={lon_b:.10f}")
print()
print(f"Distance : {t['distance']:.4f} m")
print(f"Azimuth  : {t['azimuth']:.4f}°")
print(f"dx (east): {t['dx']:+.4f} m")
print(f"dy (north): {t['dy']:+.4f} m")
print()
print(f"Translation vector A→B:  [{t['dx']:+.4f},  {t['dy']:+.4f}]  m  (east, north)")

## Compute offset coordinate (4 m in opposite direction)
Use **Mode 1** (single coordinate + azimuth) or **Mode 2** (two coordinates).

In [ ]:
# Mode 1: single coordinate + explicit azimuth
# lat_out, lon_out, az = offset_from_input(
#     lat=51.998765432100,
#     lon= 4.375123456789,
#     azimuth_deg=45.0,
# )

# Mode 2: two coordinates — azimuth derived from point1 → point2
lat_out, lon_out, az = offset_from_input(
    lat=52.008995,
    lon= 4.355610,
    lat2=52.009059,
    lon2= 4.355549,
)

print(f"Input azimuth : {az:.6f}°")
print(f"Opposite      : {(az + 180) % 360:.6f}°")
print(f"Output (4 m)  : lat={lat_out:.10f}  lon={lon_out:.10f}")

## Read GPS from MCAP

In [ ]:
GPS_FILE_PATH = r"D:\Data_gathered\2026_05_22\Rosbag\10_50_00\rosbag\rosbag_0.mcap"
TIME          = 1779439914.817100912
GPS_WINDOW    = 10.0   # seconds either side of TIME
GPS_TOPIC     = "/navsat_topic"

gps_raw = {"t": [], "lat": [], "lon": []}

gps_start_ns = int((TIME - GPS_WINDOW) * 1_000_000_000)
gps_end_ns   = int((TIME + GPS_WINDOW) * 1_000_000_000)

print(f"Reading GPS MCAP ({2*GPS_WINDOW:.0f}s window around TIME)...")
with open(GPS_FILE_PATH, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(
        topics=[GPS_TOPIC], start_time=gps_start_ns, end_time=gps_end_ns
    ):
        gps_raw["t"].append(message.publish_time / 1e9)
        gps_raw["lat"].append(ros_msg.latitude)
        gps_raw["lon"].append(ros_msg.longitude)

print(f"GPS samples: {len(gps_raw['t'])}")

## GPS coordinate at TIME

In [ ]:
times = np.array(gps_raw["t"])
lats  = np.array(gps_raw["lat"])
lons  = np.array(gps_raw["lon"])

closest_idx = int(np.argmin(np.abs(times - TIME)))

print(f"Reference TIME : {TIME:.9f}")
print(f"Closest GPS t  : {times[closest_idx]:.9f}  (Δt = {times[closest_idx] - TIME:+.6f} s)")
print(f"Coordinate     : lat={lats[closest_idx]:.10f}  lon={lons[closest_idx]:.10f}")

## Find GPS timestamp closest to target coordinate

In [ ]:
MATCH_THRESHOLD_M = 0.1  # metres

lats  = np.array(gps_raw["lat"])
lons  = np.array(gps_raw["lon"])
times = np.array(gps_raw["t"])

_, _, dists = geod.inv(
    np.full(len(lons), lon_out), np.full(len(lats), lat_out),
    lons, lats,
)
dists = np.abs(dists)

best    = int(np.argmin(dists))
matches = np.where(dists < MATCH_THRESHOLD_M)[0]

print(f"Target : lat={lat_out:.10f}  lon={lon_out:.10f}\n")

if len(matches) > 0:
    print(f"GPS within {MATCH_THRESHOLD_M} m — {len(matches)} sample(s):")
    for i in matches:
        print(f"  t={times[i]:.9f}  dist={dists[i]*100:.1f} cm  "
              f"lat={lats[i]:.10f}  lon={lons[i]:.10f}")
else:
    print(f"No GPS sample within {MATCH_THRESHOLD_M} m of target.")
    print(f"Closest: t={times[best]:.9f}  dist={dists[best]:.4f} m  "
          f"lat={lats[best]:.10f}  lon={lons[best]:.10f}")